# Triton temporal SWG attention — GPU serving primitive

This benchmark moves the **query-time temporal small-world walk + Top-16 value read** onto the GPU with Triton. FAISS is used only to build/export persistent HNSW level-0 adjacency for the quality probe; it is **not in the timed query path**.

The benchmark has two stages: (1) correctness + exact-Top16 recall on trained ML-1M temporal keys, and (2) latency scaling versus PyTorch SDPA at `L=200 / 1k / 10k`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, shutil, torch
REPO='/content/Sparsewalker'
BRANCH='agent/serving-speed-benchmark'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','faiss-cpu'],check=True)
sys.path.insert(0,f'{REPO}/src'); sys.path.insert(0,f'{REPO}/experiments')
for name in list(sys.modules):
    if name=='sparsewalker' or name.startswith('sparsewalker.'): del sys.modules[name]
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__,'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH)


## 1. Compile + correctness smoke

This uses only 16 validation users and `L=200`. It should print `REAL_QUALITY` and two `SPEED` rows. The important fail-fast fields are `triton_vs_reference_max_abs_ctx` and `triton_exact_id_row_match_rate`.


In [ ]:
import runpy, sys
SCRIPT=f'{REPO}/experiments/run_triton_temporal_swg_benchmark.py'
sys.argv=[SCRIPT,'--quality-users','16','--lengths','200','--users','1','--hops','4','--beam','16']
print('TRITON SWG SMOKE START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('TRITON SWG SMOKE END',flush=True)


## 2. Full temporal primitive benchmark

Runs the trained-key quality probe on 128 long-history users, then compares Triton SWG with PyTorch SDPA for batch-1 and batch-32 users at three history lengths. Graph construction is excluded from timing because the graph is persistent serving state.


In [ ]:
sys.argv=[SCRIPT,'--quality-users','128','--lengths','200','1000','10000','--users','1','32','--hops','4','--beam','16']
print('TRITON SWG FULL START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('TRITON SWG FULL END',flush=True)


In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_triton_swg/result.json')
r=json.loads(p.read_text())
print('QUALITY',r['quality'])
print('SPEED')
for row in r['speed']:
    print(row)
